# Extract Continuous Values Closest to Target Time After t0

**Purpose:**  
Given a set of index events (e.g., drug change dates from `build_continuous_medication_periods`) and a set of repeated clinical measurements (e.g., HbA1c, BMI, weight), find the single measurement closest to a target time point for each index event.

**This notebook contains:**
1. **`get_closest_continuous_value_after_t0`** — The main function. A general-purpose tool that finds the closest measurement to a target date within a configurable window. Works for any continuous biomarker.
2. **`enrich_index_with_timing_vars`** — A helper that adds Lancet-paper timing variables to `build_continuous_medication_periods` output. *Optional — only needed for Lancet-style dynamic windows.*
3. **`get_lancet_hba1c_baseline_and_outcome`** — A wrapper that packages the full Lancet HbA1c extraction logic (tighter baseline window, ≤61-day exclusion, dynamic outcome ceiling, 12m/6m fallback) into a single call. *Optional — only needed for Lancet replication.*

**Notebook structure:**
- **Functions** — All three function definitions
- **Example Workflows** — Copy-pasteable recipes for common use cases
- **Analysis** — Empty workspace to build your own extraction

## Imports and Data Loading

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path_to_combo = "..."

# Load the combo therapy periods output from Function 1
combo_therapy_periods = pd.read_csv(path_to_combo, parse_dates=['t0'])

# Load measurement data
# This can be any repeated continuous measurement: HbA1c, BMI, weight, eGFR, etc.
# Just ensure it has columns for patient ID, measurement date, and measurement value.
hba1c_info = pd.read_csv("...")
bmi_info = pd.read_csv("...")

combo_therapy_periods.head()

## Main Function: `get_closest_continuous_value_after_t0`

This is a general-purpose function. It simply:
1. Takes index events (patient + date)
2. Takes repeated measurements (patient + date + value)
3. Finds the closest measurement to a target date within a configurable window
4. Returns one value per index event

All Lancet-specific behavior (dynamic windows, continued therapy rules) is controlled by the **caller** through optional per-row ceiling/floor columns.

### How to adapt the function call for different biomarkers

Reminder: if you want to apply lancet logic, you must call 'enrich_index_with_timing_vars()' first, since the columns from that resulting df will be used as input into get_closest_continuous_value_after_t0()

| Scenario | target_days | days_before | days_after | max_days_from_t0_col | min_days_from_t0_col | prefer_value |
|----------|------------|-------------|------------|---------------------|---------------------|-------------|
| **Baseline HbA1c** (Lancet) | 0 | 183 | 7 | — | `'hba1c_max_lookback'` | `'min'` |
| **Baseline BMI/weight/HDL/etc.** (Lancet) | 0 | 730 | 7 | — | — | `'min'` |
| **Baseline eGFR** (Lancet) | 0 | 730 | 7 | — | — | `'max'` |
| **12-month outcome HbA1c** (Lancet) | 365 | 91 | 92 | `'maxvaliddays12m'` | — | `'min'` |
| **6-month outcome HbA1c** (Lancet) | 183 | 92 | 91 | `'maxvaliddays6m'` | — | `'min'` |
| **Simple fixed-window** (no Lancet logic) | 365 | 90 | 90 | — | — | `'min'` |

In [ ]:
def get_closest_continuous_value_after_t0(
    index_df,
    values_df,
    pat_id_col='patient_id',
    t0_col='t0',
    value_date_col='date',
    value_col='result',
    target_days_from_t0=365,
    days_before_target_date=91,
    days_after_target_date=92,
    value_name='outcome',
    max_days_from_t0_col=None,
    min_days_from_t0_col=None,
    prefer_value='min'
):
    """
    Find the closest measurement to a target date for each index event.

    This is a general-purpose function. It works with any continuous biomarker
    (HbA1c, BMI, weight, eGFR, blood pressure, etc.). All biomarker-specific
    behavior is controlled by the caller through the parameters.

    Parameters
    ----------
    index_df : pd.DataFrame
        One row per index event. Must contain pat_id_col and t0_col.
        May contain additional columns that will be preserved in the output.
    values_df : pd.DataFrame
        Repeated measurements. Must contain pat_id_col, value_date_col, value_col.
    pat_id_col : str
        Patient ID column (must exist in both DataFrames).
    t0_col : str
        Index date column in index_df.
    value_date_col : str
        Measurement date column in values_df.
    value_col : str
        Measurement value column in values_df.
    target_days_from_t0 : int
        Days after t0 that defines the "ideal" measurement date.
        Use 0 for baseline extraction, 365 for 12-month outcome, etc.
    days_before_target_date : int
        How many days before the target date the window starts.
    days_after_target_date : int
        How many days after the target date the window ends.
    value_name : str
        Prefix for output column names. For example, 'prehba1c' produces
        'prehba1c_value', 'prehba1c_date', etc. Use descriptive prefixes
        like 'prebmi', 'postweight', 'preegfr', etc.
    max_days_from_t0_col : str or None
        Optional. Column name in index_df containing a per-row ceiling
        (max days from t0 that a measurement can occur). If the ceiling
        is smaller than the default window_end, it clips the window.
        If the ceiling makes the window invalid (ceiling < window_start),
        the row gets NaN. Used for outcome extraction where continued
        therapy rules vary per patient.
    min_days_from_t0_col : str or None
        Optional. Column name in index_df containing a per-row floor
        (max allowed lookback in days, as a positive number). If the floor
        is smaller than days_before_target_date, it clips the window start.
        Used when you want to prevent selecting a measurement from before
        a certain event (e.g., a previous therapy change).
    prefer_value : str
        Tie-breaking when multiple measurements are equally close to the
        target date. 'min' picks the smallest value (default, used for most
        biomarkers). 'max' picks the largest (e.g., for eGFR where higher
        values indicate better kidney function).

    Returns
    -------
    pd.DataFrame
        NEW DataFrame with the same rows as index_df plus four new columns:
        {value_name}_value, {value_name}_date, {value_name}_days_from_target,
        {value_name}_days_from_t0. Rows with no valid measurement get NaN.
    """

    # -------------------------------------------------------------------------
    # STEP A: Preprocessing
    # -------------------------------------------------------------------------

    # Work on copies so we don't modify the caller's data
    idx = index_df.copy()
    vals = values_df[[pat_id_col, value_date_col, value_col]].copy()

    # Parse dates
    idx[t0_col] = pd.to_datetime(idx[t0_col])
    vals[value_date_col] = pd.to_datetime(vals[value_date_col], errors='coerce')

    # Drop measurement rows with missing date or value
    vals = vals.dropna(subset=[value_date_col, value_col])

    # Add a unique row ID so we can track every index row through the merge,
    # even if a patient has multiple t0 events on the same date
    idx['_row_id'] = range(len(idx))

    # -------------------------------------------------------------------------
    # STEP B: Compute the default window for each index row
    #
    # target_date is the "ideal" measurement date.
    # window_start and window_end define the acceptable range.
    #
    # Ex: For baseline (target_days=0, days_before=183, days_after=7):
    #   target_date = t0
    #   window = [t0 - 183 days, t0 + 7 days]
    #
    # Ex: For 12-month outcome (target_days=365, days_before=91, days_after=92):
    #   target_date = t0 + 365
    #   window = [t0 + 274, t0 + 457]
    # -------------------------------------------------------------------------

    idx['_target_date'] = idx[t0_col] + pd.to_timedelta(target_days_from_t0, unit='D')
    idx['_window_start'] = idx['_target_date'] - pd.to_timedelta(days_before_target_date, unit='D')
    idx['_window_end'] = idx['_target_date'] + pd.to_timedelta(days_after_target_date, unit='D')

    # -------------------------------------------------------------------------
    # STEP C: Apply per-row ceiling (optional - Lancet)
    #
    # Clips window_end per patient. Useful when the maximum valid measurement
    # date varies by patient (e.g., because they changed therapies at
    # different times).
    #
    # If you're not using dynamic windows, simply don't pass
    # max_days_from_t0_col and this step is skipped entirely.
    # -------------------------------------------------------------------------

    if max_days_from_t0_col is not None:
        ceiling_date = idx[t0_col] + pd.to_timedelta(idx[max_days_from_t0_col], unit='D')
        idx['_window_end'] = pd.DataFrame({
            'a': idx['_window_end'], 'b': ceiling_date
        }).min(axis=1)

    # -------------------------------------------------------------------------
    # STEP D: Apply per-row floor (optional - Lancet)
    #
    # Clips window_start per patient. Useful when the maximum allowed
    # lookback varies by patient (e.g., because you don't want measurements
    # from before a previous therapy change).
    #
    # min_days_from_t0_col should contain positive numbers representing the
    # maximum allowed lookback in days. E.g., if the value is 90, the
    # window_start can't be earlier than t0 - 90 days.
    #
    # If you're not using this constraint, simply don't pass
    # min_days_from_t0_col and this step is skipped entirely.
    # -------------------------------------------------------------------------

    if min_days_from_t0_col is not None:
        floor_date = idx[t0_col] - pd.to_timedelta(idx[min_days_from_t0_col], unit='D')
        idx['_window_start'] = pd.DataFrame({
            'a': idx['_window_start'], 'b': floor_date
        }).max(axis=1)

    # -------------------------------------------------------------------------
    # STEP E: Merge index events with measurements
    #
    # Inner join on patient ID creates every possible (index event, measurement)
    # pair for each patient. We then filter to the valid window in the next step.
    # -------------------------------------------------------------------------

    merged = idx.merge(vals, on=pat_id_col, how='inner')

    # -------------------------------------------------------------------------
    # STEP F: Filter to measurements within the valid window
    #
    # Also exclude rows where the ceiling made the window invalid
    # (window_end < window_start). These patients don't have a valid
    # measurement window and will get NaN in the output.
    # -------------------------------------------------------------------------

    merged = merged[
        (merged['_window_start'] <= merged['_window_end']) &
        (merged[value_date_col] >= merged['_window_start']) &
        (merged[value_date_col] <= merged['_window_end'])
    ].copy()

    # -------------------------------------------------------------------------
    # STEP G: Compute distance from target and select the best measurement
    #
    # Tie-breaking hierarchy (matches the R code):
    #   1. Closest to target date (smallest absolute distance)
    #   2. Preferred value direction (controlled by prefer_value param)
    #   3. Earliest measurement date
    # -------------------------------------------------------------------------

    merged['_abs_distance'] = (merged[value_date_col] - merged['_target_date']).dt.days.abs()

    # Determine sort order for the value column based on prefer_value
    value_ascending = (prefer_value == 'min')

    merged = merged.sort_values(
        ['_row_id', '_abs_distance', value_col, value_date_col],
        ascending=[True, True, value_ascending, True]
    )

    # Take the best (first) measurement per index row
    best = (
        merged
        .groupby('_row_id')
        .first()
        .reset_index()
    )

    # -------------------------------------------------------------------------
    # STEP H: Build output columns
    #
    # Four new columns using the value_name prefix:
    #   {value_name}_value          - the selected measurement value
    #   {value_name}_date           - date of that measurement
    #   {value_name}_days_from_target - signed days from the target date
    #   {value_name}_days_from_t0    - signed days from t0
    # -------------------------------------------------------------------------

    best = best[['_row_id', value_col, value_date_col]].copy()
    best = best.rename(columns={
        value_col: f'{value_name}_value',
        value_date_col: f'{value_name}_date'
    })

    # -------------------------------------------------------------------------
    # STEP I: Left join back to original index_df
    #
    # This guarantees every index row appears in the output. Rows with no
    # valid measurement in the window get NaN for all four output columns.
    # We build the result as a NEW DataFrame so the caller's index_df
    # is not modified.
    # -------------------------------------------------------------------------

    result = idx.merge(best, on='_row_id', how='left')

    # Compute the two distance columns from the selected measurement
    result[f'{value_name}_days_from_target'] = (
        (result[f'{value_name}_date'] - result['_target_date']).dt.days
    )
    result[f'{value_name}_days_from_t0'] = (
        (result[f'{value_name}_date'] - result[t0_col]).dt.days
    )

    # Clean up internal working columns
    internal_cols = ['_row_id', '_target_date', '_window_start', '_window_end']
    result = result.drop(columns=[c for c in internal_cols if c in result.columns])

    return result

---

# Lancet-Specific Functions

The functions below are **optional** — only needed if you want to replicate the Lancet 5-drug paper's methodology (Dennis et al., 2025). If you just want simple fixed-window extraction, skip this section entirely and go straight to the Example Workflows.

## Helper Function: Enrich Index with Timing Variables

**When to use this:**  
Call this function if you want to replicate the Lancet paper's approach, where the outcome measurement window is *dynamically shortened* per patient based on when they changed therapies. It is also needed for certain baseline constraints (e.g., the Lancet paper's rule that baseline HbA1c cannot come from before the patient's previous therapy change).

**When you can skip this:**  
If you just want a simple fixed window (e.g., "give me the closest value to 365 days post-t0, within ±90 days"), you do NOT need to call this function. Just call `get_closest_continuous_value_after_t0` directly with your desired window parameters and omit the `max_days_from_t0_col` and `min_days_from_t0_col` arguments.

In [ ]:
def enrich_index_with_timing_vars(
    combo_df,
    pat_id_col='patient_id',
    t0_col='t0',
    period_end_col='period_end_date',
    max_gap_days=183
):
    """
    Add Lancet-paper timing variables to the combo therapy periods from Function 1.

    These variables describe the temporal context of each therapy period and are
    used to construct per-patient dynamic windows for baseline and outcome extraction.

    Parameters
    ----------
    combo_df : pd.DataFrame
        Output from Function 1 (build_continuous_medication_periods).
    pat_id_col : str
        Patient ID column name.
    t0_col : str
        Column with the start date of each combo period.
    period_end_col : str
        Column with the end date (last Rx date) of each combo period.
    max_gap_days : int
        Same gap threshold used in Function 1 (default 183). AKA how long a patient can go without therapy before they are considered on a break.

    Returns
    -------
    pd.DataFrame
        Copy of combo_df with three new columns:
        - timetoaddrem_class:  days from t0 to the next combo period's t0 (NaN if none)
        - timetochange_class:  days from t0 to the effective therapy change date
        - timeprevcombo_class: days from the previous combo period's t0 to this t0 (NaN if first)
    """
    df = combo_df.copy()

    # Ensure dates are datetime
    df[t0_col] = pd.to_datetime(df[t0_col])
    df[period_end_col] = pd.to_datetime(df[period_end_col])

    # --- timetoaddrem_class ---
    # Days from this period's t0 to the next period's t0.
    # Represents when any drug was added or removed (i.e., the regimen changed).
    # NaN if this is the patient's last combo period.
    next_t0 = df.groupby(pat_id_col)[t0_col].shift(-1)
    df['timetoaddrem_class'] = (next_t0 - df[t0_col]).dt.days

    # --- timetochange_class ---
    # Days from t0 to the effective "change" date.
    # If the next combo starts within max_gap_days of the current period's end,
    # the change date is the next combo's start (seamless transition).
    # Otherwise, the change date is the current period's end (patient just stopped).
    gap_to_next = (next_t0 - df[period_end_col]).dt.days

    datechange = np.where(
        next_t0.isna() | (gap_to_next > max_gap_days),
        df[period_end_col],
        next_t0
    )
    datechange = pd.to_datetime(datechange)
    df['timetochange_class'] = (datechange - df[t0_col]).dt.days

    # --- timeprevcombo_class ---
    # Days from the previous combo period's t0 to this period's t0.
    # NaN for the patient's first combo period.
    # Can be used to constrain baseline lookback: prevents selecting a
    # measurement from before the previous therapy change, which would
    # reflect a different treatment context. (The Lancet paper applies
    # this constraint specifically for baseline HbA1c.)
    prev_t0 = df.groupby(pat_id_col)[t0_col].shift(1)
    df['timeprevcombo_class'] = (df[t0_col] - prev_t0).dt.days

    return df

## Lancet HbA1c Wrapper Function

The function below uses `enrich_index_with_timing_vars` (defined above) and `get_closest_continuous_value_after_t0` to replicate the exact HbA1c window selection and fallback logic from the Lancet 5-drug prediction paper (Dennis et al., 2025). It packages all of the following into a single call:

- **Baseline HbA1c:** 6-month lookback with a per-row floor (`timeprevcombo`) that prevents grabbing measurements from before the patient's previous therapy change
- **≤61-day exclusion:** Baselines are nulled out when the previous therapy change was too recent for HbA1c to stabilize
- **Outcome HbA1c:** Dynamic per-row ceiling (`maxvaliddays`) enforcing the continued therapy rule
- **12-month → 6-month fallback:** Uses 6-month outcome when no valid 12-month measurement exists

These rules are all **HbA1c-specific**. Other biomarkers (BMI, eGFR, HDL, etc.) use a simpler 2-year baseline lookback with no per-row constraints — see the parameter table and Example Workflows sections. The one Lancet constraint that applies to all biomarkers equally is the dynamic outcome ceiling (`maxvaliddays`), shown in Workflow A of the Example Workflows section.

### ⚠️ This does NOT fully replicate the Lancet cohort

This function only handles **measurement window selection**. The Lancet paper applies additional inclusion/exclusion criteria that are not handled here — you will need to filter separately. Key criteria not accounted for (summary):

- **Population:** Ages 18–79 only, no pre-existing ESKD
- **Drug initiations:** Only the 5 target drug classes (specific ingredients only), first instance per class, not first-line therapy, no concurrent insulin
- **Baseline values:** HbA1c must be 53–110 mmol/mol, all 9 model features must be present
- **Outcome:** Initiations with no valid outcome HbA1c are dropped for the prediction model (though retained for long-term outcome analyses)

The output should be treated as **raw extracted values** requiring post-processing to match the Lancet analysis cohort.

In [ ]:
def get_lancet_hba1c_baseline_and_outcome(
    index_df,
    values_df,
    pat_id_col='patient_id',
    t0_col='t0',
    value_date_col='date',
    value_col='result'
):
    """
    Extract baseline and outcome HbA1c following the Lancet 5-drug paper methodology.

    This wraps multiple calls to get_closest_continuous_value_after_t0 with the
    exact parameters used in 02_mm_baseline_biomarkers.R and 03_mm_response_biomarkers.R,
    plus the 12m/6m fallback from IODC_POC_TL_fixed.ipynb.

    Parameters
    ----------
    index_df : pd.DataFrame
        Output from enrich_index_with_timing_vars(). Must contain timing columns:
        timetoaddrem_class, timetochange_class, timeprevcombo_class.
    values_df : pd.DataFrame
        HbA1c measurement data with pat_id_col, value_date_col, value_col.
    pat_id_col, t0_col, value_date_col, value_col : str
        Column name overrides.

    Returns
    -------
    pd.DataFrame
        New DataFrame with all original columns plus:
        - prehba1c_value, prehba1c_date, prehba1c_days_from_target, prehba1c_days_from_t0
        - posthba1c_value, posthba1c_date, posthba1c_days_from_target, posthba1c_days_from_t0
        - posthba1c_source ('12m' or '6m')
        - hba1c_response (outcome - baseline, NaN if either is missing)
    """

    # -----------------------------------------------------------------
    # BASELINE: 6-month lookback, timeprevcombo floor, <=61-day exclusion
    # -----------------------------------------------------------------

    df = index_df.copy()

    # Per-row floor: can't look back further than the previous therapy change
    df['_hba1c_max_lookback'] = (
        df['timeprevcombo_class'].fillna(183).clip(upper=183)
    )

    df = get_closest_continuous_value_after_t0(
        index_df=df,
        values_df=values_df,
        pat_id_col=pat_id_col,
        t0_col=t0_col,
        value_date_col=value_date_col,
        value_col=value_col,
        target_days_from_t0=0,
        days_before_target_date=183,
        days_after_target_date=7,
        value_name='prehba1c',
        min_days_from_t0_col='_hba1c_max_lookback',
        prefer_value='min'
    )

    # Null out baselines where previous therapy change was <= 61 days ago
    # (HbA1c hasn't had time to stabilize from the prior change)
    unstable = (
        df['timeprevcombo_class'].notna() &
        (df['timeprevcombo_class'] <= 61)
    )
    for col in ['prehba1c_value', 'prehba1c_date',
                'prehba1c_days_from_target', 'prehba1c_days_from_t0']:
        df.loc[unstable, col] = np.nan

    # Clean up internal column
    df = df.drop(columns=['_hba1c_max_lookback'])

    # -----------------------------------------------------------------
    # OUTCOME: 12-month with dynamic ceiling, 6-month fallback
    # -----------------------------------------------------------------

    # 12-month ceiling: min(timetoaddrem, timetochange+91, 457)
    addrem_cap_12m = df['timetoaddrem_class'].fillna(457)
    change_cap_12m = df['timetochange_class'].fillna(457 - 91) + 91
    df['_maxvaliddays12m'] = np.minimum(
        np.minimum(addrem_cap_12m, change_cap_12m), 457
    )

    # 6-month ceiling: min(timetoaddrem, timetochange+91, 274)
    addrem_cap_6m = df['timetoaddrem_class'].fillna(274)
    change_cap_6m = df['timetochange_class'].fillna(274 - 91) + 91
    df['_maxvaliddays6m'] = np.minimum(
        np.minimum(addrem_cap_6m, change_cap_6m), 274
    )

    # 12-month outcome
    outcome_12m = get_closest_continuous_value_after_t0(
        index_df=df,
        values_df=values_df,
        pat_id_col=pat_id_col,
        t0_col=t0_col,
        value_date_col=value_date_col,
        value_col=value_col,
        target_days_from_t0=365,
        days_before_target_date=91,
        days_after_target_date=92,
        value_name='posthba1c',
        max_days_from_t0_col='_maxvaliddays12m',
        prefer_value='min'
    )

    # 6-month fallback
    outcome_6m = get_closest_continuous_value_after_t0(
        index_df=df,
        values_df=values_df,
        pat_id_col=pat_id_col,
        t0_col=t0_col,
        value_date_col=value_date_col,
        value_col=value_col,
        target_days_from_t0=183,
        days_before_target_date=92,
        days_after_target_date=91,
        value_name='posthba1c',
        max_days_from_t0_col='_maxvaliddays6m',
        prefer_value='min'
    )

    # -----------------------------------------------------------------
    # COMBINE: prefer 12-month, fill gaps from 6-month
    # -----------------------------------------------------------------

    # Start from the baseline df (which has prehba1c columns)
    # and add outcome columns from the 12-month results
    result = df.copy()
    outcome_col_names = ['posthba1c_value', 'posthba1c_date',
                         'posthba1c_days_from_target', 'posthba1c_days_from_t0']

    for col in outcome_col_names:
        result[col] = outcome_12m[col].values

    result['posthba1c_source'] = np.where(
        result['posthba1c_value'].notna(), '12m', None
    )

    # Fill from 6-month where 12-month is missing
    missing = result['posthba1c_value'].isna()
    for col in outcome_col_names:
        result.loc[missing, col] = outcome_6m[col].values[missing]

    result.loc[missing, 'posthba1c_source'] = np.where(
        result.loc[missing, 'posthba1c_value'].notna(), '6m', None
    )

    # -----------------------------------------------------------------
    # RESPONSE
    # -----------------------------------------------------------------
    result['hba1c_response'] = (
        result['posthba1c_value'] - result['prehba1c_value']
    )

    # Clean up internal columns
    result = result.drop(columns=['_maxvaliddays12m', '_maxvaliddays6m'], errors='ignore')

    return result

---

# Example Workflows

These examples show how to extract multiple biomarkers and combine them into a single DataFrame — the most common real-world usage pattern.

### Workflow A: Lancet-compliant extraction (HbA1c + other baselines)

Uses `get_lancet_hba1c_baseline_and_outcome` for HbA1c, plus simple calls for other biomarkers.

```python
# Step 1: Enrich index with timing variables
index_df = enrich_index_with_timing_vars(combo_therapy_periods)

# Step 2: Lancet HbA1c (baseline + outcome + response, all in one call)
result_df = get_lancet_hba1c_baseline_and_outcome(
    index_df=index_df,
    values_df=hba1c_info,
    value_date_col='date',
    value_col='result'
)
# result_df now has: prehba1c_*, posthba1c_*, posthba1c_source, hba1c_response

# Step 3: Baseline eGFR (2-year lookback, prefer max)
result_df = get_closest_continuous_value_after_t0(
    index_df=result_df, values_df=egfr_info,
    value_date_col='date', value_col='egfr_value',
    target_days_from_t0=0, days_before_target_date=730, days_after_target_date=7,
    value_name='preegfr', prefer_value='max'
)

# Step 4: Baseline BMI (2-year lookback, prefer min)
result_df = get_closest_continuous_value_after_t0(
    index_df=result_df, values_df=bmi_info,
    value_date_col='date', value_col='bmi_value',
    target_days_from_t0=0, days_before_target_date=730, days_after_target_date=7,
    value_name='prebmi', prefer_value='min'
)

# result_df now has: prehba1c_*, posthba1c_*, hba1c_response, preegfr_*, prebmi_*
```

### Workflow B: Simple fixed-window extraction (no Lancet constraints)

No enrichment needed. Every biomarker gets the same straightforward treatment.

```python
# Baseline HbA1c (1-year lookback)
result_df = get_closest_continuous_value_after_t0(
    index_df=combo_therapy_periods, values_df=hba1c_info,
    value_date_col='date', value_col='result',
    target_days_from_t0=0, days_before_target_date=365, days_after_target_date=7,
    value_name='prehba1c', prefer_value='min'
)

# Outcome HbA1c (12-month, +/-90 day window, no dynamic ceiling, no fallback)
result_df = get_closest_continuous_value_after_t0(
    index_df=result_df, values_df=hba1c_info,
    value_date_col='date', value_col='result',
    target_days_from_t0=365, days_before_target_date=90, days_after_target_date=90,
    value_name='posthba1c', prefer_value='min'
)

# Baseline eGFR (1-year lookback, prefer max)
result_df = get_closest_continuous_value_after_t0(
    index_df=result_df, values_df=egfr_info,
    value_date_col='date', value_col='egfr_value',
    target_days_from_t0=0, days_before_target_date=365, days_after_target_date=7,
    value_name='preegfr', prefer_value='max'
)

# Baseline BMI (1-year lookback)
result_df = get_closest_continuous_value_after_t0(
    index_df=result_df, values_df=bmi_info,
    value_date_col='date', value_col='bmi_value',
    target_days_from_t0=0, days_before_target_date=365, days_after_target_date=7,
    value_name='prebmi', prefer_value='min'
)

# Compute response
result_df['hba1c_response'] = result_df['posthba1c_value'] - result_df['prehba1c_value']

# result_df now has: prehba1c_*, posthba1c_*, hba1c_response, preegfr_*, prebmi_*
```

**Key difference:** Workflow A uses `get_lancet_hba1c_baseline_and_outcome` which handles the tighter window, `timeprevcombo` floor, ≤61-day exclusion, dynamic ceiling, and 6-month fallback automatically. Workflow B skips all of that — every biomarker gets the same simple treatment.

In both cases, each call chains from the previous `result_df`, so all columns accumulate into a single DataFrame naturally.

In [ ]:
bmi_info.head()

---

# Analysis

Follow one of the workflows above, or build your own using the functions defined in this notebook.

---

## Export Results

In [ ]:
file_name = ''  # <-- set your output filename here
# result_df.to_csv(file_name, index=False)
# print(f"Exported {len(result_df)} rows to {file_name}")